# RF-DETR-M — UICVD / uicvd16
Thesis experiment: `rfdetr_m_uicvd_uicvd16`

**Datasets required (add via right panel):** `uidet-uicvd16`, `uidet-code`

**Secret required:** `WANDB_API_KEY`

In [ ]:
!pip install -q rfdetr supervision pycocotools faster-coco-eval wandb pyyaml

In [ ]:
import os, sys, json, shutil, glob
from pathlib import Path

WORKING = Path('/kaggle/working')

def find_input(slug):
    for pattern in [f'/kaggle/input/{slug}',
                    f'/kaggle/input/datasets/*/{slug}',
                    f'/kaggle/input/datasets/*/*/{slug}']:
        matches = glob.glob(pattern)
        if matches:
            return Path(matches[0])
    raise FileNotFoundError(f"Dataset '{slug}' not found under /kaggle/input/")

DATA_INPUT = find_input('uidet-uicvd16')
CODE_INPUT = find_input('uidet-code')

print('DATA_INPUT:', DATA_INPUT)
print('CODE_INPUT:', CODE_INPUT)

sys.path.insert(0, str(CODE_INPUT / 'src'))
import uidet
print('uidet imported OK:', uidet.__file__)

In [ ]:
import yaml

actual_images      = DATA_INPUT / 'images'
actual_annotations = DATA_INPUT / 'annotations'

work_data = WORKING / 'uicvd' / 'uicvd16'
work_data.mkdir(parents=True, exist_ok=True)

images_link = work_data / 'images'
if images_link.is_symlink(): images_link.unlink()
images_link.symlink_to(actual_images)
print('images link ok:', images_link.exists())

ann_dir = work_data / 'annotations'
ann_dir.mkdir(exist_ok=True)
for f in actual_annotations.glob('*.json'):
    shutil.copy2(f, ann_dir / f.name)
    print('copied:', f.name)

with open(DATA_INPUT / 'data.yaml') as f:
    data_cfg = yaml.safe_load(f)
data_cfg['path'] = str(work_data)
data_yaml_path = work_data / 'data.yaml'
with open(data_yaml_path, 'w') as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)
print('data.yaml written OK')

In [ ]:
SPLIT_MAP = {"train": "train", "val": "valid", "test": "test"}
rfdetr_root = WORKING.parent / 'prepared_rfdetr' / 'uicvd__uicvd16'
print('Writing RF-DETR layout to:', rfdetr_root)

for our_split, rf_split in SPLIT_MAP.items():
    out_split = rfdetr_root / rf_split
    out_split.mkdir(parents=True, exist_ok=True)

    ann_files = list(actual_annotations.glob(f'*_{our_split}.json'))
    if not ann_files:
        print(f'WARN: no annotation json for {our_split}, skipping')
        continue

    coco = json.loads(ann_files[0].read_text())
    for im in coco['images']:
        im['file_name'] = Path(im['file_name']).name
    (out_split / '_annotations.coco.json').write_text(json.dumps(coco))

    src_imgs = actual_images / our_split
    n = 0
    for src in src_imgs.iterdir():
        if not src.is_file(): continue
        dst = out_split / src.name
        if dst.exists() or dst.is_symlink(): continue
        try:
            dst.symlink_to(src.resolve())
        except OSError:
            shutil.copy2(src, dst)
        n += 1
    print(f'  {our_split} -> {rf_split}: {n} images staged')

from uidet.models.rfdetr import _resolve_dataset_dir
resolved = _resolve_dataset_dir(data_yaml_path)
print('Resolved:', resolved)
print('Match:', resolved == rfdetr_root)

In [ ]:
from uidet.models.base import TrainConfig, build_detector
from uidet.train import get_class_names

EXP_NAME   = 'rfdetr_m_uicvd_uicvd16'
MODEL_NAME = 'rfdetr_m'
DATASET    = 'uicvd'
TAXONOMY   = 'uicvd16'

out_dir = WORKING / 'results_v2' / EXP_NAME
out_dir.mkdir(parents=True, exist_ok=True)

class_names = get_class_names(data_yaml_path)
print(f'{len(class_names)} classes: {class_names}')

cfg = TrainConfig(
    name=EXP_NAME,
    out_dir=out_dir,
    data_yaml=data_yaml_path,
    coco_val_json=ann_dir / f'{DATASET}_{TAXONOMY}_val.json',
    coco_test_json=ann_dir / f'{DATASET}_{TAXONOMY}_test.json',
    epochs=80,
    batch=2,
    imgsz=640,
    lr0=0.0001,
    seed=42,
    device='0',
    workers=4,
    amp=True,
    extra={
        'grad_accum_steps': 16,
        'resolution': 576,
        'weight_decay': 0.0001,
        'early_stopping': True,
        'early_stopping_patience': 15,
        'use_ema': True,
    },
    wandb_project='uidet-thesis',
    wandb_entity=None,
    wandb_tags=['kaggle', 'rfdetr', DATASET, TAXONOMY],
)
print('Config OK ->', out_dir)

In [ ]:
import wandb

try:
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret('WANDB_API_KEY'))
    print('WandB logged in via Kaggle secret')
except Exception as e:
    print(f'Secret not found ({e}), falling back...')
    wandb.login()

os.environ['WANDB_PROJECT'] = 'uidet-thesis'
os.environ['WANDB_NAME']    = cfg.name

from uidet.train import _wandb_init, _wandb_log_summary, _wandb_finish

cfg_dict = {
    'name': EXP_NAME, 'model': MODEL_NAME,
    'dataset': DATASET, 'taxonomy': TAXONOMY,
    'wandb_project': 'uidet-thesis', 'wandb_entity': None, 'wandb_tags': cfg.wandb_tags,
}
wandb_run = _wandb_init(cfg_dict, cfg, class_names)

detector = build_detector(MODEL_NAME, num_classes=len(class_names), class_names=class_names)
print(f'Training {MODEL_NAME} on {DATASET}/{TAXONOMY} ({len(class_names)} classes)...')

weights = detector.train(cfg)
print('Best weights:', weights)

In [ ]:
import time
from uidet.utils.metrics import coco_evaluate, detections_to_coco, format_metrics_table

detector.load(weights)

gt_path = ann_dir / f'{DATASET}_{TAXONOMY}_test.json'
gt      = json.loads(gt_path.read_text())
items   = [(im['id'], work_data / im['file_name']) for im in gt['images']]
name_to_cat_id = {c: i + 1 for i, c in enumerate(class_names)}

predictions, t0 = [], time.perf_counter()
for i in range(0, len(items), 4):
    chunk = items[i:i+4]
    batch_dets = detector.predict_batch([p for _, p in chunk], conf=0.001, iou=0.6)
    for image_id, dets in zip([iid for iid, _ in chunk], batch_dets):
        predictions.extend(detections_to_coco(dets, image_id, name_to_cat_id))
dt = time.perf_counter() - t0

eval_dir = out_dir / 'coco_eval_test'
metrics  = coco_evaluate(str(gt_path), predictions, eval_dir)
metrics['inference_seconds'] = dt
metrics['inference_fps']     = len(items) / dt
(eval_dir / 'metrics.json').write_text(json.dumps(metrics, indent=2))
print(format_metrics_table(metrics))

In [ ]:
if wandb_run is not None:
    wandb.log({
        'test/mAP':   metrics['mAP'],
        'test/mAP50': metrics['mAP50'],
        'test/mAP75': metrics['mAP75'],
        **{f'test/AP/{cls}':   v['AP']   for cls, v in metrics['per_class'].items()},
        **{f'test/AP50/{cls}': v['AP50'] for cls, v in metrics['per_class'].items()},
    })
    print('COCO metrics logged to WandB')
_wandb_log_summary(wandb_run, metrics, split='test')
_wandb_finish(wandb_run)

import shutil
shutil.make_archive(str(WORKING / EXP_NAME), 'zip', str(out_dir))
print('Download:', str(WORKING / EXP_NAME) + '.zip')
print(json.dumps(metrics, indent=2))